# Classical Reasoning Engine

The notebook contains the complete implementation, dataset loaders, evaluation, and CSV generation. It uses symbolic inference; it does not call a generative model or create replacement benchmark examples.

Public evaluation uses RuleTaker (`jise/ruletaker`, train), FOLIO (validation), and both published CounterBench JSON files. Accuracy is reported only over successfully scored records, with coverage and unsupported counts shown separately.


In [ ]:
%pip -q install datasets huggingface_hub

Note: you may need to restart the kernel to use updated packages.


## Configuration

Set `HF_TOKEN` in the environment only if you have accepted the gated FOLIO repository terms. The loader falls back to the original Yale-LILY FOLIO validation export. Set the private file paths below to evaluate a labelled private development file or to create predictions for an unlabelled private test file.

In [ ]:
import os
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN")
DATASET_LIMIT = 50
PRIVATE_DEV_PATH = "private_dev.jsonl"
PRIVATE_TEST_PATH = "private_test.jsonl"


## Reasoning engine and dataset adapters

All inference code is defined in this notebook cell. RuleTaker records are converted to ground atoms and Horn rules; FOLIO formal annotations are grounded and checked by CNF/DPLL refutation; CounterBench records are processed by the deterministic causal simulator. Unsupported records are never silently converted to `False`.

In [ ]:
from __future__ import annotations
import csv
import json
import re
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Set, Tuple
Atom = Tuple[str, ...]
Rule = Tuple[Tuple[Atom, ...], Atom]
Literal = Tuple[Atom, bool]
Clause = frozenset[Literal]

def is_var(term: str) -> bool:
    return term.startswith('?')

def unify(a: Atom, b: Atom, bindings: Optional[dict]=None) -> Optional[dict]:
    if len(a) != len(b) or a[0] != b[0]:
        return None
    result = dict(bindings or {})
    for x, y in zip(a[1:], b[1:]):
        while is_var(x) and x in result:
            x = result[x]
        while is_var(y) and y in result:
            y = result[y]
        if x == y:
            continue
        if is_var(x):
            result[x] = y
        elif is_var(y):
            result[y] = x
        else:
            return None
    return result

def substitute(atom: Atom, bindings: Mapping[str, str]) -> Atom:
    values = []
    for value in atom:
        while value.startswith('?') and value in bindings:
            value = bindings[value]
        values.append(value)
    return tuple(values)

def match_body(body: Sequence[Atom], facts: Set[Atom]):

    def visit(i: int, env: dict):
        if i == len(body):
            yield env
            return
        pattern = substitute(body[i], env)
        for fact in tuple(facts):
            updated = unify(pattern, fact, env)
            if updated is not None:
                yield from visit(i + 1, updated)
    yield from visit(0, {})

def forward_chain(facts: Sequence[Atom], rules: Sequence[Rule], query: Optional[Atom]=None):
    known: Set[Atom] = set((tuple(x) for x in facts))
    trace: List[str] = []
    if query is not None and query in known:
        return (known, trace)
    changed = True
    while changed:
        changed = False
        for body, head in rules:
            for env in match_body(body, known):
                derived = substitute(head, env)
                if any((is_var(x) for x in derived[1:])) or derived in known:
                    continue
                known.add(derived)
                changed = True
                trace.append(f'{body} with {env} -> {derived}')
                if query is not None and derived == query:
                    return (known, trace)
    return (known, trace)

def as_atom(value: Sequence[str]) -> Atom:
    if not isinstance(value, (list, tuple)) or not value:
        raise ValueError(f'Not an atom: {value!r}')
    return tuple((str(x) for x in value))

def rule_from_record(item) -> Rule:
    body, head = item
    return (tuple((as_atom(x) for x in body)), as_atom(head))

def horn_predict(record: Mapping) -> bool:
    facts = [as_atom(x) for x in record.get('facts', [])]
    rules = [rule_from_record(x) for x in record.get('rules', [])]
    query = as_atom(record['query'])
    derived, _ = forward_chain(facts, rules, query)
    return query in derived

def _sentences(text: str) -> List[str]:
    return [x.strip().rstrip('.?! ') for x in re.split('(?<=[.!?])\\s+', text) if x.strip()]

def _clean_name(text: str) -> str:
    text = re.sub('^(?:the|a|an)\\s+', '', text.strip(), flags=re.I)
    text = re.sub('[^\\w -]', '', text).strip().lower()
    return re.sub('\\s+', '_', text)

def parse_rt_atom(sentence: str) -> Optional[Atom]:
    s = sentence.strip().rstrip('.?! ')
    s = re.sub('^(?:the|a|an)\\s+', '', s, flags=re.I)
    if not s:
        return None
    m = re.match('^(someone|something|anyone|anything|they|it)\\s+(?:is|are|was|were)\\s+(not\\s+)?(.+)$', s, flags=re.I)
    if m:
        subject = '?x'
        predicate = re.sub('\\s+', '_', m.group(3).strip().lower())
        if m.group(2):
            predicate = '~' + predicate
    else:
        m = re.match('^(.+?)\\s+(?:is|are|was|were)\\s+(not\\s+)?(.+)$', s, flags=re.I)
        if m:
            subject = m.group(1).strip()
            predicate = re.sub('\\s+', '_', m.group(3).strip().lower())
            if m.group(2):
                predicate = '~' + predicate
        else:
            generic = re.match('^(someone|something|anyone|anything|they|it)\\s+(.+)$', s, flags=re.I)
            if generic:
                subject = '?x'
                predicate = generic.group(2).strip().lower()
            else:
                words = s.split()
                if len(words) < 2:
                    return None
                subject_words = 1
                if len(words) >= 3 and words[0][:1].isupper() and words[1][:1].isupper():
                    subject_words = 2
                subject = ' '.join(words[:subject_words])
                predicate = ' '.join(words[subject_words:]).strip().lower()
            predicate = re.sub('\\s+', '_', predicate)
    subject = _clean_name(subject) if not subject.startswith('?') else subject
    predicate = re.sub('[^\\w~-]', '', predicate)
    if not subject or not predicate:
        return None
    return (predicate, subject)

def parse_rt_rule(sentence: str) -> Optional[Rule]:
    s = sentence.strip().rstrip('.?! ')
    patterns = ['^(?:if|when)\\s+(.+?)\\s*,?\\s+then\\s+(.+)$', '^(?:when|assuming|given that|based on)\\s+(.+?)\\s*,\\s*(?:we can say|we can conclude|it follows that|it is evident that|it is clear that)\\s+(.+)$', '^(?:given that)\\s+(.+?)\\s*,\\s*(?:it follows|we can conclude|we can say)\\s+that\\s+(.+)$']
    for pattern in patterns:
        m = re.match(pattern, s, flags=re.I)
        if not m:
            continue
        antecedents = re.split('\\s+and\\s+', m.group(1).strip(), flags=re.I)
        body = [parse_rt_atom(part) for part in antecedents]
        head = parse_rt_atom(m.group(2))
        if body and all(body) and head:
            return (tuple(body), head)
    return None

def parse_ruletaker_record(row: Mapping) -> dict:
    context = str(row.get('context', ''))
    statement = str(row.get('statement', row.get('question', '')))
    query = parse_rt_atom(statement)
    if query is None:
        raise ValueError('could not parse RuleTaker statement')
    facts_text = context
    rules_text = ''
    m = re.search('\\bFacts\\s*:\\s*(.*?)\\bRules\\s*:\\s*(.*)$', context, flags=re.I | re.S)
    if m:
        facts_text, rules_text = (m.group(1), m.group(2))
    facts: List[Atom] = []
    rules: List[Rule] = []
    for sentence in _sentences(facts_text):
        sentence = re.sub('^Facts\\s*:\\s*', '', sentence, flags=re.I)
        atom = parse_rt_atom(sentence)
        if atom is None:
            raise NotImplementedError(f'unsupported RuleTaker fact template: {sentence!r}')
        facts.append(atom)
    for sentence in _sentences(rules_text):
        sentence = re.sub('^Rules\\s*:\\s*', '', sentence, flags=re.I)
        rule = parse_rt_rule(sentence)
        if rule is None:
            raise NotImplementedError(f'unsupported RuleTaker rule template: {sentence!r}')
        rules.append(rule)
    if not facts and (not rules):
        raise NotImplementedError('no RuleTaker facts or rules were recognised')
    return {'id': str(row.get('id', row.get('row_id', ''))), 'dataset': 'RuleTaker', 'facts': [list(x) for x in facts], 'rules': [[[list(a) for a in body], list(head)] for body, head in rules], 'query': list(query), 'label': row.get('flag', row.get('label')), 'depth': row.get('depth')}
TOKEN_RE = re.compile('\\s*(<->|↔|->|→|⇒|∀|∃|¬|~|!|∧|&|∨|\\||⊕|\\(|\\)|,|[A-Za-z_][A-Za-z0-9_-]*)')

class FormulaParser:

    def __init__(self, text: str):
        self.tokens = TOKEN_RE.findall(text)
        compact = TOKEN_RE.sub('', text)
        if compact.strip():
            raise ValueError(f'Unsupported FOL syntax near: {compact.strip()[:30]}')
        self.i = 0

    def peek(self):
        return self.tokens[self.i] if self.i < len(self.tokens) else None

    def take(self, expected=None):
        token = self.peek()
        if token is None:
            raise ValueError('unexpected end of formula')
        if expected is not None and token != expected:
            raise ValueError(f'expected {expected!r}, got {token!r}')
        self.i += 1
        return token

    def parse(self):
        node = self.iff()
        if self.peek() is not None:
            raise ValueError(f'unexpected token {self.peek()!r}')
        return node

    def iff(self):
        left = self.implication()
        while self.peek() in {'↔', '<->'}:
            self.take()
            left = ('iff', left, self.implication())
        return left

    def implication(self):
        left = self.disjunction()
        if self.peek() in {'→', '->', '⇒'}:
            self.take()
            return ('imp', left, self.implication())
        return left

    def disjunction(self):
        left = self.xor()
        while self.peek() in {'∨', '|', 'or'}:
            self.take()
            left = ('or', left, self.xor())
        return left

    def xor(self):
        left = self.conjunction()
        while self.peek() == '⊕':
            self.take()
            left = ('xor', left, self.conjunction())
        return left

    def conjunction(self):
        left = self.unary()
        while self.peek() in {'∧', '&', 'and'}:
            self.take()
            left = ('and', left, self.unary())
        return left

    def unary(self):
        if self.peek() in {'¬', '~', '!'}:
            self.take()
            return ('not', self.unary())
        if self.peek() in {'∀', '∃'}:
            quantifier = 'forall' if self.take() == '∀' else 'exists'
            variable = self.take()
            if not re.fullmatch('[a-zA-Z][\\w-]*', variable):
                raise ValueError('invalid quantified variable')
            body = self.unary()
            return (quantifier, variable, body)
        if self.peek() == '(':
            self.take('(')
            node = self.iff()
            self.take(')')
            return node
        predicate = self.take()
        if not re.fullmatch('[A-Za-z_][A-Za-z0-9_-]*', predicate):
            raise ValueError(f'expected predicate, got {predicate!r}')
        self.take('(')
        args = [self.take()]
        while self.peek() == ',':
            self.take(',')
            args.append(self.take())
        self.take(')')
        return ('atom', predicate, tuple(args))

def parse_formula(text: str):
    return FormulaParser(str(text).strip().rstrip('.')).parse()

def collect_constants(node, bound: Optional[Set[str]]=None) -> Set[str]:
    bound = set(bound or ())
    kind = node[0]
    if kind == 'atom':
        return {x for x in node[2] if x not in bound}
    if kind in {'forall', 'exists'}:
        return collect_constants(node[2], bound | {node[1]})
    result: Set[str] = set()
    for child in node[1:]:
        if isinstance(child, tuple):
            result.update(collect_constants(child, bound))
    return result

def ground_formula(node, domain: Sequence[str], env: Optional[dict]=None):
    env = dict(env or {})
    kind = node[0]
    if kind == 'atom':
        return ('atom', node[1], tuple((env.get(x, x) for x in node[2])))
    if kind == 'forall':
        pieces = []
        for value in domain:
            next_env = dict(env)
            next_env[node[1]] = value
            pieces.append(ground_formula(node[2], domain, next_env))
        if not pieces:
            raise ValueError('empty domain for universal quantifier')
        out = pieces[0]
        for piece in pieces[1:]:
            out = ('and', out, piece)
        return out
    if kind == 'exists':
        pieces = []
        for value in domain:
            next_env = dict(env)
            next_env[node[1]] = value
            pieces.append(ground_formula(node[2], domain, next_env))
        if not pieces:
            raise ValueError('empty domain for existential quantifier')
        out = pieces[0]
        for piece in pieces[1:]:
            out = ('or', out, piece)
        return out
    if kind == 'not':
        return ('not', ground_formula(node[1], domain, env))
    return (kind, ground_formula(node[1], domain, env), ground_formula(node[2], domain, env))

def to_nnf(node, negated: bool=False):
    kind = node[0]
    if kind == 'atom':
        atom = (node[1],) + tuple(node[2])
        return ('lit', atom, negated)
    if kind == 'not':
        return to_nnf(node[1], not negated)
    if kind in {'and', 'or'}:
        op = kind
        if negated:
            op = 'or' if op == 'and' else 'and'
        return (op, to_nnf(node[1], negated), to_nnf(node[2], negated))
    if kind == 'imp':
        if negated:
            return ('and', to_nnf(node[1], False), to_nnf(node[2], True))
        return ('or', to_nnf(node[1], True), to_nnf(node[2], False))
    if kind == 'iff':
        a, b = (node[1], node[2])
        if negated:
            return ('or', ('and', to_nnf(a, False), to_nnf(b, True)), ('and', to_nnf(a, True), to_nnf(b, False)))
        return ('and', ('or', to_nnf(a, True), to_nnf(b, False)), ('or', to_nnf(b, True), to_nnf(a, False)))
    if kind == 'xor':
        a, b = (node[1], node[2])
        if negated:
            return ('and', ('or', to_nnf(a, True), to_nnf(b, False)), ('or', to_nnf(a, False), to_nnf(b, True)))
        return ('and', ('or', to_nnf(a, False), to_nnf(b, False)), ('or', to_nnf(a, True), to_nnf(b, True)))
    raise ValueError(f'unsupported formula node: {kind}')

def cnf(node) -> List[Clause]:
    kind = node[0]
    if kind == 'lit':
        return [frozenset({(node[1], node[2])})]
    left, right = (cnf(node[1]), cnf(node[2]))
    if kind == 'and':
        return left + right
    if kind == 'or':
        combined = []
        for a in left:
            for b in right:
                clause = set(a) | set(b)
                if any(((atom, not sign) in clause for atom, sign in clause)):
                    continue
                combined.append(frozenset(clause))
        return combined
    raise ValueError(f'invalid NNF operator: {kind}')

def _simplify_cnf(clauses: List[Clause], chosen: Literal):
    opposite = (chosen[0], not chosen[1])
    result = []
    for clause in clauses:
        if chosen in clause:
            continue
        reduced = clause - {opposite}
        if not reduced:
            return None
        result.append(reduced)
    return result

def cnf_satisfiable(clauses: List[Clause]) -> bool:

    def search(current: List[Clause]) -> bool:
        current = list(current)
        while True:
            if any((not c for c in current)):
                return False
            if not current:
                return True
            units = [next(iter(c)) for c in current if len(c) == 1]
            if units:
                lit = units[0]
                reduced = _simplify_cnf(current, lit)
                if reduced is None:
                    return False
                current = reduced
                continue
            all_lits = set().union(*current)
            pure = next((lit for lit in all_lits if (lit[0], not lit[1]) not in all_lits), None)
            if pure is not None:
                reduced = _simplify_cnf(current, pure)
                if reduced is None:
                    return False
                current = reduced
                continue
            break
        chosen = next(iter(min(current, key=len)))
        left = _simplify_cnf(current, chosen)
        if left is not None and search(left):
            return True
        right = _simplify_cnf(current, (chosen[0], not chosen[1]))
        return right is not None and search(right)
    return search(clauses)

def count_existentials(node) -> int:
    kind = node[0]
    if kind == 'exists':
        return 1 + count_existentials(node[2])
    if kind == 'forall':
        return count_existentials(node[2])
    if kind in {'atom'}:
        return 0
    if kind == 'not':
        return count_existentials(node[1])
    return count_existentials(node[1]) + count_existentials(node[2])

def folio_predict(row: Mapping) -> bool:
    premises = row.get('premises-FOL') or row.get('premises_fol') or []
    if isinstance(premises, str):
        premises = [x.strip() for x in premises.splitlines() if x.strip()]
    conclusion = row.get('conclusion-FOL') or row.get('conclusion_fol')
    if not premises or not conclusion:
        raise ValueError('FOLIO row is missing premises-FOL or conclusion-FOL')
    premise_asts = [parse_formula(x) for x in premises]
    query_ast = parse_formula(str(conclusion))
    constants: Set[str] = set()
    all_asts = premise_asts + [query_ast]
    for ast in all_asts:
        constants.update(collect_constants(ast))
    witness_count = sum((count_existentials(ast) for ast in all_asts))
    constants.update((f'__witness_{i}' for i in range(witness_count)))
    domain = sorted(constants) or ['domain_object']
    clauses: List[Clause] = []
    for ast in premise_asts:
        ground = ground_formula(ast, domain)
        clauses.extend(cnf(to_nnf(ground)))
    ground_query = ground_formula(query_ast, domain)
    clauses.extend(cnf(to_nnf(ground_query, negated=True)))
    return not cnf_satisfiable(clauses)

def _event_name(value: str) -> str:
    return re.sub('[^a-z0-9_-]', '', value.strip().lower())

def parse_causal_rules(given_info: str):
    text = re.sub('^\\s*we know that\\s+', '', given_info.strip(), flags=re.I)
    text = re.split('\\bwe observed\\b', text, maxsplit=1, flags=re.I)[0]
    text = re.sub('\\s+[A-Za-z][\\w-]*\\s*~\\s*Bern\\([^)]*\\)\\.?', '', text, flags=re.I)
    pieces = re.split(',\\s*(?:and\\s+)?', text)
    rules = []
    for piece in pieces:
        piece = re.sub('^and\\s+', '', piece.strip(), flags=re.I).rstrip('.')
        piece = re.sub('^we know that\\s+', '', piece, flags=re.I)
        m = re.match('^(.+?)\\s+(together\\s+)?causes?\\s+(not\\s+)?([\\w-]+)$', piece, flags=re.I)
        if not m:
            continue
        cause_text, together, negative_head, effect = m.groups()
        effect_name = _event_name(effect)
        if not effect_name:
            continue
        if re.search('\\s+or\\s+', cause_text, flags=re.I):
            bodies = [(_event_name(x),) for x in re.split('\\s+or\\s+', cause_text, flags=re.I)]
        elif re.search('\\s+and\\s+', cause_text, flags=re.I):
            bodies = [tuple((_event_name(x) for x in re.split('\\s+and\\s+', cause_text, flags=re.I)))]
        else:
            bodies = [(_event_name(cause_text),)]
        for body in bodies:
            if body and all(body):
                rules.append((body, effect_name, not bool(negative_head)))
    return rules

def _counterbench_observations(text: str) -> Dict[str, bool]:
    observations: Dict[str, bool] = {}
    for match in re.finditer('\\bwe\\s+observed\\s+([^.!?]+)', text, flags=re.I):
        part = re.sub('^\\s*that\\s+', '', match.group(1), flags=re.I)
        part = re.sub('\\b(?:occurs?|occurred)\\b', '', part, flags=re.I)
        for piece in re.split('\\s+and\\s+|,\\s*', part, flags=re.I):
            item = re.fullmatch('\\s*(not\\s+)?([\\w-]+)(?:\\s+(not))?\\s*', piece, flags=re.I)
            if item:
                is_negative = bool(item.group(1) or item.group(3))
                observations[_event_name(item.group(2))] = not is_negative
    return observations

def _counterbench_assignments(question: str):
    interventions: Dict[str, bool] = {}
    baseline: Dict[str, bool] = {}
    for m in re.finditer('\\b(?:assume|suppose)\\s+(not\\s+)?([\\w-]+)', question, flags=re.I):
        interventions[_event_name(m.group(2))] = not bool(m.group(1))
    m = re.search('\\bif\\s+(.+?)(?:\\s+instead\\s+of\\s+|\\?)', question, flags=re.I)
    if m:
        for piece in re.split('\\s+and\\s+', m.group(1), flags=re.I):
            item = re.fullmatch('\\s*(not\\s+)?([\\w-]+)\\s*', piece, flags=re.I)
            if item:
                interventions[_event_name(item.group(2))] = not bool(item.group(1))
    m = re.search('\\binstead\\s+of\\s+(.+?)\\??$', question.strip(), flags=re.I)
    if m:
        for piece in re.split('\\s+and\\s+', m.group(1), flags=re.I):
            item = re.fullmatch('\\s*(not\\s+)?([\\w-]+)\\s*', piece, flags=re.I)
            if item:
                baseline[_event_name(item.group(2))] = not bool(item.group(1))
    return (interventions, baseline)

def _simulate_causal_model(rules, root_values: Mapping[str, bool], interventions: Mapping[str, bool]):
    true_events = {name for name, value in root_values.items() if value}
    false_events = {name for name, value in root_values.items() if not value}
    by_effect = defaultdict(list)
    for body, effect, value in rules:
        by_effect[effect].append((body, value))
    for effect, effect_rules in by_effect.items():
        if effect not in interventions and effect not in root_values and all((not value for _, value in effect_rules)):
            true_events.add(effect)
    for event, value in interventions.items():
        true_events.discard(event)
        false_events.discard(event)
        (true_events if value else false_events).add(event)
    changed = True
    passes = 0
    while changed:
        changed = False
        passes += 1
        if passes > max(2, len(by_effect) * 2 + 1):
            return None
        for effect, effect_rules in by_effect.items():
            if effect in interventions or effect in root_values:
                continue
            positive = any((value and all((x in true_events for x in body)) for body, value in effect_rules))
            negative = any((not value and all((x in true_events for x in body)) for body, value in effect_rules))
            if positive and negative:
                return None
            if positive:
                if effect not in true_events:
                    true_events.add(effect)
                    false_events.discard(effect)
                    changed = True
            elif negative:
                if effect not in false_events:
                    false_events.add(effect)
                    true_events.discard(effect)
                    changed = True
            elif all((not value for _, value in effect_rules)):
                if effect not in true_events:
                    true_events.add(effect)
                    false_events.discard(effect)
                    changed = True
            elif effect not in false_events:
                false_events.add(effect)
                true_events.discard(effect)
                changed = True
    return (true_events, false_events)

def counterbench_predict(row: Mapping) -> bool:
    given_info = str(row.get('given_info', ''))
    question = str(row.get('question', ''))
    rules = parse_causal_rules(given_info)
    if not rules:
        raise ValueError('no causal rules were recognised')
    target_match = re.search('\\bwould\\s+([\\w-]+)\\s+occur\\b', question, flags=re.I)
    if not target_match:
        raise ValueError('could not identify CounterBench outcome variable')
    target = _event_name(target_match.group(1))
    interventions, baseline = _counterbench_assignments(question)
    observations = _counterbench_observations(question + ' ' + given_info)
    probabilities = {_event_name(name): float(prob) for name, prob in re.findall('\\b([A-Za-z][\\w-]*)\\s*~\\s*Bern\\(([0-9]*\\.?[0-9]+)\\)', given_info, flags=re.I)}
    all_variables = {target} | set(observations) | set(interventions) | set(baseline)
    effects = set()
    for body, effect, _ in rules:
        effects.add(effect)
        all_variables.add(effect)
        all_variables.update(body)
    roots = sorted(all_variables - effects)
    if len(roots) > 16:
        raise NotImplementedError('too many exogenous variables for exhaustive abduction')
    options = []
    for root in roots:
        p = probabilities.get(root)
        if p is not None and p <= 0:
            options.append((False,))
        elif p is not None and p >= 1:
            options.append((True,))
        else:
            options.append((False, True))
    import itertools
    outcomes = set()
    consistent_worlds = 0
    for values in itertools.product(*options):
        root_values = dict(zip(roots, values))
        actual_world = _simulate_causal_model(rules, root_values, {})
        if actual_world is None:
            continue
        actual_true, actual_false = actual_world
        actual_state = {name: True for name in actual_true}
        actual_state.update({name: False for name in actual_false})
        evidence = dict(observations)
        evidence.update(baseline)
        if any((actual_state.get(name, False) != value for name, value in evidence.items())):
            continue
        consistent_worlds += 1
        counterfactual = _simulate_causal_model(rules, root_values, interventions)
        if counterfactual is None:
            continue
        outcomes.add(target in counterfactual[0])
    if consistent_worlds == 0 or not outcomes:
        raise ValueError('no factual causal assignment satisfies the observations')
    if len(outcomes) > 1:
        raise NotImplementedError('observations do not determine a unique counterfactual answer')
    return outcomes.pop()

def gold_bool(value) -> bool:
    text = str(value).strip().lower().replace('_', ' ').replace('-', ' ')
    text = ' '.join(text.split())
    if text in {'true', '1', 'yes', 'entailment', 'entailed'}:
        return True
    if text in {'false', '0', 'no', 'not entailment', 'not entailed', 'uncertain', 'unknown', 'contradiction', 'neutral'}:
        return False
    raise ValueError(f'unknown gold label: {value!r}')

def predict_record(row: Mapping) -> bool:
    dataset = str(row.get('dataset', '')).lower()
    if dataset == 'ruletaker' or ('context' in row and ('statement' in row or 'question' in row)):
        return horn_predict(parse_ruletaker_record(row))
    if dataset == 'folio' or ('premises-FOL' in row and 'conclusion-FOL' in row):
        return folio_predict(row)
    if dataset == 'counterbench' or ('given_info' in row and 'question' in row):
        return counterbench_predict(row)
    if 'facts' in row or 'rules' in row:
        return horn_predict(row)
    raise ValueError(f'no evaluator available for record fields={sorted(row.keys())}')

def load_jsonl(path: str) -> List[dict]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    if p.suffix.lower() == '.json':
        data = json.loads(p.read_text(encoding='utf-8'))
        return data if isinstance(data, list) else data.get('data', data.get('records', []))
    rows = []
    with p.open(encoding='utf-8') as f:
        for line_number, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'{path}:{line_number}: invalid JSON') from exc
    return rows

def evaluate(rows: Iterable[Mapping]) -> dict:
    totals = defaultdict(lambda: {"correct": 0, "scored": 0, "total": 0, "unsupported": 0, "seconds": 0.0})
    errors = []
    overall = {"correct": 0, "scored": 0, "total": 0, "unsupported": 0, "seconds": 0.0}
    for i, row in enumerate(rows):
        dataset = str(row.get("dataset", "unknown"))
        group_name = dataset
        if row.get("split"):
            group_name += "/" + str(row["split"])
        elif dataset.lower() == "counterbench" and row.get("type"):
            group_name += "/" + str(row["type"])
        group = totals[group_name]
        group["total"] += 1
        overall["total"] += 1
        try:
            expected = gold_bool(row.get("label", row.get("answer", row.get("flag"))))
        except Exception as exc:
            group["unsupported"] += 1
            overall["unsupported"] += 1
            if len(errors) < 15:
                errors.append({"id": row.get("id", i), "dataset": group_name, "unsupported": str(exc)})
            continue
        start = time.perf_counter()
        try:
            actual = predict_record(row)
        except Exception as exc:
            elapsed = time.perf_counter() - start
            group["seconds"] += elapsed
            overall["seconds"] += elapsed
            group["unsupported"] += 1
            overall["unsupported"] += 1
            if len(errors) < 15:
                errors.append({"id": row.get("id", i), "dataset": group_name, "unsupported": str(exc)})
            continue
        elapsed = time.perf_counter() - start
        group["seconds"] += elapsed
        overall["seconds"] += elapsed
        ok = int(actual == expected)
        group["correct"] += ok
        group["scored"] += 1
        overall["correct"] += ok
        overall["scored"] += 1
        if not ok and len(errors) < 15:
            errors.append({"id": row.get("id", i), "dataset": group_name,
                           "gold": expected, "predicted": actual,
                           "type": row.get("type", "")})

    
    def summary(x):
        scored, total = x["scored"], x["total"]

        return {
            "correct": x["correct"],
            "scored": scored,
            "total": total,
            "unsupported": x["unsupported"],
            "accuracy_percent": (
                100 * x["correct"] / scored if scored else None
            ),
            "coverage_percent": (
                100 * scored / total if total else 0.0
            ),
            "strict_accuracy_percent": (
                100 * x["correct"] / total if total else 0.0
            ),
            "inference_seconds": x["seconds"],
            "average_ms": (
                1000 * x["seconds"] / total if total else 0.0
            ),
        }

    return {
        "overall": summary(overall),
        "datasets": {
            k: summary(v) for k, v in totals.items()
        },
        "examples": errors
    }    

def write_submission(rows: Iterable[Mapping], path: str='submission.csv'):
    predictions = []
    for index, row in enumerate(rows):
        try:
            prediction = predict_record(row)
        except Exception as exc:
            raise ValueError(f'Test record {index} is unsupported: {exc}') from exc
        predictions.append((row.get('id', index), str(prediction)))
    with open(path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['id', 'label'])
        writer.writerows(predictions)
    return len(predictions)


## Public dataset downloaders

Only the named public sources are used. A failed download raises an error; no synthetic examples or locally generated substitute benchmark is created.

In [ ]:
import json
import os
import urllib.request
from pathlib import Path

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'Saved {len(rows)} records -> {path}')

def load_json_url(url):
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=30) as response:
        text = response.read().decode('utf-8')
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            for key in ('data', 'records', 'examples', 'questions'):
                if isinstance(obj.get(key), list):
                    return obj[key]
            return [obj]
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    return []

def fetch_ruletaker(splits, limit=None):
    try:
        from datasets import load_dataset
    except ImportError as exc:
        raise RuntimeError('Install the datasets package: pip install datasets') from exc
    results = {}
    for split in splits:
        print(f'Downloading jise/ruletaker split={split} ...')
        dataset = load_dataset('jise/ruletaker', split=split)
        rows = []
        for i, row in enumerate(dataset):
            if limit and len(rows) >= limit:
                break
            rows.append({'id': f'ruletaker_{split}_{i}', 'dataset': 'RuleTaker', 'split': split, 'context': row['context'], 'statement': row['statement'], 'depth': row.get('depth'), 'flag': row['flag']})
        results[split] = rows
        write_jsonl(f'ruletaker_{split}.jsonl', rows)
    return results

def fetch_folio(limit=None, token=None):
    print('Downloading FOLIO validation split ...')
    rows = None
    try:
        from datasets import load_dataset
        dataset = load_dataset('yale-nlp/FOLIO', split='validation', token=token or os.environ.get('HF_TOKEN'))
        rows = list(dataset)
        source = 'Hugging Face yale-nlp/FOLIO'
    except Exception as hf_error:
        url = 'https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-validation.jsonl'
        try:
            rows = load_json_url(url)
            source = 'official Yale-LILY GitHub validation mirror'
        except Exception as mirror_error:
            raise RuntimeError('Could not download FOLIO. Accept the dataset conditions at https://huggingface.co/datasets/yale-nlp/FOLIO and set HF_TOKEN, or check internet access. No example fallback was used.') from mirror_error
    if limit:
        rows = rows[:limit]
    output = []
    for i, row in enumerate(rows):
        if not row.get('premises-FOL') or not row.get('conclusion-FOL') or row.get('label') is None:
            continue
        output.append({'id': str(row.get('example_id', row.get('id', f'folio_{i}'))), 'dataset': 'FOLIO', 'premises-FOL': row['premises-FOL'], 'conclusion-FOL': row['conclusion-FOL'], 'label': row['label'], 'source': source})
    write_jsonl('folio_validation.jsonl', output)
    print(f'FOLIO source: {source}')
    return output

def fetch_counterbench(limit=None):
    base = 'https://huggingface.co/datasets/CounterBench/CounterBench/resolve/main/'
    filenames = ['data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json']
    all_rows = []
    for filename in filenames:
        print(f'Downloading CounterBench/{filename} ...')
        rows = load_json_url(base + filename)
        cleaned = []
        for i, row in enumerate(rows):
            if not isinstance(row, dict):
                continue
            if not all((k in row for k in ('given_info', 'question', 'answer'))):
                continue
            item = dict(row)
            item['id'] = f"cb_{('backdoor' if 'backdoor' in filename else 'alpha')}_{row.get('question_id', i)}"
            item['dataset'] = 'CounterBench'
            item['source_file'] = filename
            cleaned.append(item)
        if limit:
            cleaned = cleaned[:limit]
        outname = 'counterbench_alpha.jsonl' if 'alpha' in filename else 'counterbench_backdoor.jsonl'
        write_jsonl(outname, cleaned)
        all_rows.extend(cleaned)
    return all_rows


## Public benchmark evaluation

The notebook prints accuracy, coverage, unsupported counts, and total/average inference time. A low-coverage accuracy must not be described as full-dataset accuracy.

In [ ]:
ruletaker_splits = fetch_ruletaker(
    ["train", "test", "ood_test"],
    limit=DATASET_LIMIT
)
folio_rows = fetch_folio(limit=DATASET_LIMIT, token=HF_TOKEN)
counterbench_rows = fetch_counterbench(limit=DATASET_LIMIT)
public_dev_rows = (
    list(ruletaker_splits["ood_test"])
    + list(folio_rows)
    + list(counterbench_rows)
)
if not public_dev_rows:
    raise RuntimeError("No public benchmark records were downloaded.")

with open("dev.jsonl", "w", encoding="utf-8") as file:
    for row in public_dev_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

public_metrics = evaluate(public_dev_rows)
print("PUBLIC DATASET EVALUATION")
print(f"Scored accuracy: {public_metrics['overall']['accuracy_percent']}")
print(f"Coverage: {public_metrics['overall']['coverage_percent']:.2f}%")
print(f"Scored: {public_metrics['overall']['scored']} / {public_metrics['overall']['total']}")
print(f"Unsupported: {public_metrics['overall']['unsupported']}")
print(f"Total inference time: {public_metrics['overall']['inference_seconds']:.6f} s")
print(f"Average inference time: {public_metrics['overall']['average_ms']:.4f} ms/record")

print("\nPER-DATASET RESULTS")
for name, item in sorted(public_metrics["datasets"].items()):
    print(
        f"{name}: accuracy={item['accuracy_percent']}%, "
        f"coverage={item['coverage_percent']:.2f}%, "
        f"scored={item['scored']}/{item['total']}, "
        f"unsupported={item['unsupported']}, "
        f"inference={item['inference_seconds']:.6f} s"
    )

Path("evaluation_results.json").write_text(
    json.dumps(public_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)


c:\Users\vaibh\Downloads\final_submission_checked\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 50 records -> ruletaker_train.jsonl
Saved 50 records -> ruletaker_test.jsonl
Saved 50 records -> ruletaker_ood_test.jsonl
Saved 50 records -> folio_validation.jsonl
FOLIO source: official Yale-LILY GitHub validation mirror
Saved 50 records -> counterbench_alpha.jsonl
Saved 50 records -> counterbench_backdoor.jsonl
PUBLIC DATASET EVALUATION
Scored accuracy: 98.9795918367347
Coverage: 98.00%
Scored: 196 / 200
Unsupported: 4
Total inference time: 0.045943 s
Average inference time: 0.2297 ms/record

PER-DATASET RESULTS
CounterBench/backdoor: accuracy=100.0%, coverage=100.00%, scored=50/50, unsupported=0, inference=0.004631 s
CounterBench/basic: accuracy=100.0%, coverage=100.00%, scored=40/40, unsupported=0, inference=0.004435 s
CounterBench/conditional: accuracy=100.0%, coverage=100.00%, scored=10/10, unsupported=0, inference=0.001186 s
FOLIO: accuracy=95.65217391304348%, coverage=92.00%, scored=46/50, unsupported=4, inference=0.021931 s
RuleTaker/ood_test: accuracy=100.0%, coverage=

2617

## Optional private-dataset evaluation

Place labelled private records in `private_dev.jsonl` using one of the supported record formats. The same reasoner is used; no private examples are generated or inferred from labels.

In [ ]:
if Path(PRIVATE_DEV_PATH).exists():
    private_dev_rows = load_jsonl(PRIVATE_DEV_PATH)
    private_metrics = evaluate(private_dev_rows)
    print("PRIVATE DEVELOPMENT DATA")
    print(f"Accuracy: {private_metrics['overall']['accuracy_percent']}%")
    print(f"Coverage: {private_metrics['overall']['coverage_percent']:.2f}%")
    print(f"Total inference time: {private_metrics['overall']['inference_seconds']:.6f} s")
    Path("private_evaluation_results.json").write_text(
        json.dumps(private_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
    )
else:
    print(f"No labelled private development file found at {PRIVATE_DEV_PATH}; skipped private accuracy.")


No labelled private development file found at private_dev.jsonl; skipped private accuracy.


## Generate `submission.csv`

If `private_test.jsonl` exists, it is used. Otherwise the notebook generates a non-empty demonstration CSV from the public RuleTaker test split. Private test rows may be source-native RuleTaker/FOLIO/CounterBench records or generic structured records with `facts`, `rules`, and `query`. Every row must be supported; otherwise CSV creation stops with an explicit error.

In [ ]:
if Path(PRIVATE_TEST_PATH).exists():
    test_rows = load_jsonl(PRIVATE_TEST_PATH)
    output_source = PRIVATE_TEST_PATH
else:
    test_rows = ruletaker_splits["test"]
    output_source = "jise/ruletaker test split"

prediction_count = write_submission(test_rows, "submission.csv")
print(f"Created submission.csv from {output_source}")
print(f"Predictions written: {prediction_count}")


Created submission.csv from jise/ruletaker test split
Predictions written: 50


### Generic private record format

Each JSONL line should contain an `id`, a `facts` list, a `rules` list, a `query`, and a `label` only for evaluation. Each rule is `[body_atoms, head_atom]`; each atom is a list whose first item is its predicate. Variables begin with `?`.

```json
{"id":"ex1","facts":[["Young","Bob"]],"rules":[[[["Young","?x"]],["Smart","?x"]]],"query":["Smart","Bob"],"label":true}
```
